In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import joblib

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()
rf = joblib.load('../backend/models/random_forest_model.pkl')
importance_df = pd.read_csv('../data/processed/feature_importance.csv')

print("✅ Loaded")

✅ Loaded


In [2]:
# Cell 2 — Pick a fraud transaction to explain
fraud_indices = np.where(y_test == 1)[0]
sample_idx = fraud_indices[0]

transaction = X_test.iloc[sample_idx]
actual_label = y_test[sample_idx]

print("Transaction index:", sample_idx)
print("Actual label:", "FRAUD" if actual_label == 1 else "LEGIT")

Transaction index: 840
Actual label: FRAUD


In [3]:
# Cell 3 — Get prediction
pred = rf.predict([transaction])[0]
prob = rf.predict_proba([transaction])[0]

print("Predicted:", "FRAUD" if pred == 1 else "LEGIT")
print(f"Fraud probability: {prob[1]*100:.2f}%")
print(f"Legit probability: {prob[0]*100:.2f}%")

Predicted: FRAUD
Fraud probability: 94.00%
Legit probability: 6.00%


c:\Users\Constantine\Desktop\FraudLens\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Constantine\Desktop\FraudLens\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [4]:
# Cell 4 — Explain the transaction
def explain_transaction(transaction, importance_df, top_n=5):
    explanation = []

    for _, row in importance_df.head(top_n).iterrows():
        feature = row['feature']
        importance = row['importance']
        value = transaction[feature]

        explanation.append({
            'feature': feature,
            'value': round(value, 4),
            'importance': round(importance, 4),
            'contribution': round(value * importance, 4)
        })

    return pd.DataFrame(explanation)

explanation = explain_transaction(transaction, importance_df)
print(explanation)

  feature   value  importance  contribution
0     V14 -6.1743      0.2219       -1.3700
1     V10 -4.8811      0.1092       -0.5328
2      V4  2.3245      0.1076        0.2502
3     V17 -6.5365      0.0828       -0.5412
4     V12 -4.6864      0.0817       -0.3830


In [5]:
# Cell 5 — Risk score (0-100)
def compute_risk_score(fraud_prob):
    return round(fraud_prob * 100, 2)

risk_score = compute_risk_score(prob[1])
print(f"\nRisk Score: {risk_score}/100")

if risk_score >= 70:
    print("🔴 HIGH RISK")
elif risk_score >= 40:
    print("🟡 MEDIUM RISK")
else:
    print("🟢 LOW RISK")


Risk Score: 94.0/100
🔴 HIGH RISK


In [6]:
# Cell 6 — Full explanation output
print("\n========== FRAUDLENS EXPLANATION ==========")
print(f"Predicted:  {'FRAUD' if pred == 1 else 'LEGIT'}")
print(f"Risk Score: {risk_score}/100")
print(f"\nTop contributing features:")
for _, row in explanation.iterrows():
    print(f"  {row['feature']}: value={row['value']} | importance={row['importance']}")
print("===========================================")


========== FRAUDLENS EXPLANATION ==========
Predicted:  FRAUD
Risk Score: 94.0/100

Top contributing features:
  V14: value=-6.1743 | importance=0.2219
  V10: value=-4.8811 | importance=0.1092
  V4: value=2.3245 | importance=0.1076
  V17: value=-6.5365 | importance=0.0828
  V12: value=-4.6864 | importance=0.0817
